In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor,HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
#from catboost import CatBoostRegressor
import pickle as pc

In [2]:
Dataset = pd.read_csv("insurance_pre.csv")
Dataset = pd.get_dummies(Dataset, dtype = "int", drop_first = True)

In [3]:
Independent = Dataset [['age', 'bmi', 'children', 'sex_male', 'smoker_yes']]
Dependent = Dataset [['charges']]
X_Train,X_Test,Y_Train,Y_Test = train_test_split (Independent,Dependent,test_size = 0.3, random_state = 0)

In [4]:
models = {
    "Mulitple_Linear_Regression" : LinearRegression(),
    "Support_Vector_Regression" :  SVR(kernel = "linear", C= 1000, epsilon = 0.1),
    "Decision_Tree_Regressor" :  DecisionTreeRegressor(criterion = "squared_error", splitter = "best", max_features = None),
    "Random_Forest_Regressor" :  RandomForestRegressor(n_estimators = 50, max_features= None , criterion = "poisson" ,random_state = 0),
    "Ada_Boost_Regressor" : AdaBoostRegressor(estimator = DecisionTreeRegressor(max_depth =3), n_estimators = 100, learning_rate = 0.1, random_state = 42),
    "Gradient_Boosting_Regressor" : GradientBoostingRegressor(n_estimators = 200, learning_rate = .05),
    "HistGradient_Boosting_Regressor" : HistGradientBoostingRegressor(learning_rate = 0.05, max_leaf_nodes = 50,random_state = 42, max_iter = 300,verbose=0),
    "XG_Boosting_Regressor" : XGBRegressor(n_estimators = 500, learning_rate = 0.05, max_depth = 6,verbose=0),
    "Light_GBM_Regressor" : LGBMRegressor(n_estimators=200, learning_rate = 1,num_leaves = 10,verbose=0)
    #"Cat_Boosting_Regressor" : CatBoostRegressor()
}

In [5]:
result = {}
high_score = 0
best_model = None
Final_model = None
for name, model in models.items():
    model.fit(X_Train, Y_Train)
    Y_Pred = model.predict(X_Test)
    result[name] = r2_score(Y_Test, Y_Pred)
    
print("Find the result below : \n" )

for model_name, score in result.items():
    print(f"{model_name}: {score:.4f}")
    
    if score >= high_score:
        high_score = score
        best_model = model_name
        Final_model = model
        
print(f"\nThe highest reported R² score model is: {best_model} with score: {high_score:.4f}")             

E:\Anaconda\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
E:\Anaconda\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
E:\Anaconda\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
E:\Anaconda\Lib\site-packages\sklearn\ensemble\_gb.py:691: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, wa

Find the result below : 

Mulitple_Linear_Regression: 0.7895
Support_Vector_Regression: 0.6340
Decision_Tree_Regressor: 0.6834
Random_Forest_Regressor: 0.8488
Ada_Boost_Regressor: 0.8563
Gradient_Boosting_Regressor: 0.8838
HistGradient_Boosting_Regressor: 0.8586
XG_Boosting_Regressor: 0.8219
Light_GBM_Regressor: 0.7879

The highest reported R² score model is: Gradient_Boosting_Regressor with score: 0.8838


In [6]:
pc.dump(Final_model,open(f"{best_model}_model.sav",'wb'))
print(f"Best Model saved as {best_model}_model.sav")

Best Model saved as Gradient_Boosting_Regressor_model.sav


In [7]:
Loaded_file = pc.load(open(f"{best_model}_model.sav",'rb'))

# Create DataFrame with same column names used during training
new_data = pd.DataFrame({
'age': [50],
'bmi': [30.970],
'children': [3],
'sex_male': [1],
'smoker_yes': [0]
})

Result = Loaded_file.predict(new_data)
Result

array([11444.25357502])